# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s-safwan/ml-internship-flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis:
One row represents exactly one unique content page per day (identified by content_hash_id and report_date).

Time Window:
We are analyzing a single mid-panel month, specifically March 2026 (2026-03-01 to 2026-03-31), to safely iterate without touching the final sealed test month.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

1. Context:
content_hash_id, report_date (Used only for identification, never given to the model).
2. Features (5):

gsc_impressions: Knowable visibility of the page in search before the prediction.

gsc_avg_position: Knowable historical ranking power.

ga4_sessions: Knowable overall site traffic.

scroll_events: Knowable user interaction metric.

sessions_direct: Knowable non-search traffic.

3. Label / Proxy:
has_clicks (Derived mathematically as gsc_clicks > 0. This proxy tells us if the page is successfully generating search traffic).
4. Excluded:
sessions_organic (Excluded because it is a direct data leak/trap. Organic sessions in GA4 represent the same underlying action as a GSC click. Giving this to the model allows it to read the answer directly rather than predicting it).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from google.colab import userdata
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# --- 1. SETUP CONNECTION ---
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

hf_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# --- 2. VERIFICATION QUERIES ---
print("--- VERIFICATION QUERIES ---")

# Query 1: The Grain Check
q1 = f"SELECT content_hash_id, report_date, COUNT(*) as c FROM read_parquet('{hf_path}') GROUP BY content_hash_id, report_date HAVING c > 1 LIMIT 5"
print("\n1. Grain Check (Should return 0 rows if grain is 1 page per day):")
display(con.execute(q1).df())

# Query 2: Counts and Time Window
q2 = f"SELECT COUNT(*) as total_rows, MIN(report_date) as start_date, MAX(report_date) as end_date FROM read_parquet('{hf_path}')"
print("\n2. Row Count and Date Span (Confirming March 2026):")
display(con.execute(q2).df())

# Query 3: Availability (IS TRUE Check)
q3 = f"SELECT COUNT(*) as ga4_surviving_rows FROM read_parquet('{hf_path}') WHERE ga4_data_available IS TRUE"
print("\n3. Availability (Rows where GA4 data IS TRUE):")
display(con.execute(q3).df())

# --- 3. THE FEATURE FRAME & TRAP (LEAKAGE) EXPERIMENT ---
print("\n--- ML TRAP EXPERIMENT ---")

q_features = f"""
SELECT
    gsc_impressions,
    gsc_avg_position,
    ga4_sessions,
    scroll_events,
    sessions_direct,
    (gsc_clicks > 0) as has_clicks, -- LABEL
    sessions_organic -- TRAP
FROM read_parquet('{hf_path}')
WHERE ga4_data_available IS TRUE
LIMIT 10000
"""
df_features = con.execute(q_features).df().dropna()

# Model WITH the Trap
X_with_trap = df_features[['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'scroll_events', 'sessions_direct', 'sessions_organic']]
y = df_features['has_clicks']
X_train, X_test, y_train, y_test = train_test_split(X_with_trap, y, random_state=42)
model = DecisionTreeClassifier(max_depth=5)
model.fit(X_train, y_train)
print(f"Score WITH the Trap (sessions_organic included): {accuracy_score(y_test, model.predict(X_test)):.3f}")

# Model WITHOUT the Trap (Honest)
X_honest = df_features[['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'scroll_events', 'sessions_direct']]
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_honest, y, random_state=42)
model.fit(X_train_h, y_train_h)
print(f"Honest Score (Trap removed): {accuracy_score(y_test_h, model.predict(X_test_h)):.3f}")

--- VERIFICATION QUERIES ---

1. Grain Check (Should return 0 rows if grain is 1 page per day):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,c



2. Row Count and Date Span (Confirming March 2026):


,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31



3. Availability (Rows where GA4 data IS TRUE):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_surviving_rows
0,413966



--- ML TRAP EXPERIMENT ---
Score WITH the Trap (sessions_organic included): 0.726
Honest Score (Trap removed): 0.693


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. Unbalanced History & GSC-Only Early Rows:
Our data suffers from uneven history. Often, Google Search Console (GSC) is connected before Google Analytics (GA4). This means early rows might have search data but completely missing on-page session data, leading to unbalanced records.

2. Survival Bias:
We are only observing pages that survived and still exist in the warehouse by March 2026. Pages that decayed to zero and were deleted earlier are invisible to us, skewing our understanding of a true "decline".

3. Window Overlaps:
Daily snapshots can overlap with rolling averages, meaning an event happening today might mathematically pollute the features of tomorrow, creating subtle time-leakage.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.